# Data Generation

This notebook generates synthetic ad tech sample data and saves it as CSV files to a Databricks volume for ingestion.

In [ ]:
import uuid
from datetime import datetime, timedelta
import random
import pandas as pd
import os

# Define volume path (adjust catalog/schema/volume as needed)
volume_path = '/Volumes/adtech_demo/adtech_platform/generated_datasets/'
os.makedirs(volume_path, exist_ok=True)

# Generate campaigns
campaigns = [
    {
        'campaign_id': f'cmp_{i:03d}',
        'campaign_name': f'Holiday Launch {i}',
        'advertiser': random.choice(['MediaCo', 'StreamMax', 'SportPlay', 'EntertainmentX']),
        'start_date': (datetime(2025, 11, 1) + timedelta(days=7 * i)).date(),
        'end_date': datetime(2025, 12, 31).date(),
        'budget': round(random.uniform(10000, 250000), 2),
        'target_audience': random.choice(['GenZ', 'Millennials', 'Families', 'Gamers']),
        'created_at': datetime.now()
    }
    for i in range(1, 6)
]

# Generate impressions, clicks, conversions, costs
impressions = []
clicks = []
conversions = []
costs = []

for campaign in campaigns:
    for j in range(200):
        imp_id = str(uuid.uuid4())
        event_ts = datetime(2025, 11, 15) + timedelta(minutes=random.randint(0, 86400))
        view_duration = max(0.0, random.gauss(12, 8))
        viewability = view_duration >= 1.0
        audience = random.choice(['Premium', 'Casual', 'Active', 'Core'])
        impressions.append({
            'impression_id': imp_id,
            'campaign_id': campaign['campaign_id'],
            'publisher': random.choice(['video_site', 'mobile_app', 'streaming_service']),
            'placement': random.choice(['pre-roll', 'mid-roll', 'banner', 'native']),
            'event_timestamp': event_ts,
            'viewability': viewability,
            'view_duration_seconds': round(view_duration, 2),
            'audience_segment': audience
        })
        if random.random() < 0.15:
            click_id = str(uuid.uuid4())
            click_ts = event_ts + timedelta(seconds=random.randint(5, 300))
            cost = round(random.uniform(0.25, 5.0), 2)
            clicks.append({
                'click_id': click_id,
                'impression_id': imp_id,
                'campaign_id': campaign['campaign_id'],
                'click_timestamp': click_ts,
                'cost': cost,
                'device': random.choice(['mobile', 'desktop', 'tablet']),
                'geo': random.choice(['US', 'UK', 'CA', 'AU'])
            })
            if random.random() < 0.35:
                conversions.append({
                    'conversion_id': str(uuid.uuid4()),
                    'click_id': click_id,
                    'campaign_id': campaign['campaign_id'],
                    'conversion_timestamp': click_ts + timedelta(minutes=random.randint(2, 90)),
                    'revenue': round(cost * random.uniform(1.5, 4.0), 2),
                    'conversion_type': random.choice(['video_watch_complete', 'signup', 'in_app_purchase'])
                })
    costs.append({
        'campaign_id': campaign['campaign_id'],
        'cost_date': campaign['start_date'],
        'daily_cost': round(campaign['budget'] / 15, 2),
        'channel': random.choice(['display', 'video', 'social', 'search'])
    })

# Save to CSVs in volume
pd.DataFrame(campaigns).to_csv(volume_path + 'campaigns.csv', index=False)
pd.DataFrame(impressions).to_csv(volume_path + 'impressions.csv', index=False)
pd.DataFrame(clicks).to_csv(volume_path + 'clicks.csv', index=False)
pd.DataFrame(conversions).to_csv(volume_path + 'conversions.csv', index=False)
pd.DataFrame(costs).to_csv(volume_path + 'costs.csv', index=False)

print('Sample data generated and saved to volume:', volume_path)
print('Files:', os.listdir(volume_path))